

# Environment check

**Course: Agentic Artificial Intelligence and Multi-Agent Systems**
Fundacion AI Granada · 13, 14 and 16 July 2026 · 10:00-12:00

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/montevive/agentic-ai-course/blob/main/00-environment-check.ipynb)

This notebook checks that your environment is ready for the hands-on classes. Run it end to end (*Run > Run All Cells* menu, or *Runtime > Run all* in Colab) and check the final result.

If something fails and you can't fix it, email us at **chema@montevive.ai** before Monday and we'll sort it out together.

> **Colab:** if you open it in Google Colab you don't need to install anything on your machine, but you will have to add your API keys in the *Secrets* panel (key icon 🔑 in the sidebar) with the names `ANTHROPIC_API_KEY`, `OPENAI_API_KEY` or `GOOGLE_API_KEY`.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q -r https://raw.githubusercontent.com/montevive/agentic-ai-course/main/requirements.txt
print(f"Python {sys.version.split()[0]}" + (" · Google Colab" if IN_COLAB else " · local environment"))

## 1 · Python version

We need **Python 3.11 or higher** (3.12 recommended).

In [ ]:
python_ok = sys.version_info >= (3, 11)
print(("✅" if python_ok else "❌") + f" Python {sys.version.split()[0]}"
      + ("" if python_ok else " -> Python 3.11 or higher is required"))

## 2 · Course packages

We check that the packages in `requirements.txt` are installed. If any is missing, run locally:

```bash
pip install -r requirements.txt
```

In [ ]:
from importlib import metadata

PACKAGES = [
    "pydantic-ai", "smolagents", "openai-agents", "langgraph", "langchain",
    "crewai", "openai", "anthropic", "google-genai", "chromadb", "mem0ai",
    "fastmcp", "mcp", "nvidia-nat", "langfuse", "tiktoken", "rich", "python-dotenv",
]

missing = []
for pkg in PACKAGES:
    try:
        print(f"✅ {pkg:<16} {metadata.version(pkg)}")
    except metadata.PackageNotFoundError:
        missing.append(pkg)
        print(f"❌ {pkg:<16} not installed")

packages_ok = not missing
print()
print("All packages installed." if packages_ok
      else f"{len(missing)} packages missing: {', '.join(missing)}")

## 3 · API keys

For the exercises you need **at least one** API key from these providers (each exercise will indicate which one to use, and we'll provide access during the training to anyone who needs it):

| Provider | Variable | Where to get it |
|---|---|---|
| Anthropic (recommended) | `ANTHROPIC_API_KEY` | https://console.anthropic.com/ |
| OpenAI | `OPENAI_API_KEY` | https://platform.openai.com/ |
| Google Gemini | `GOOGLE_API_KEY` | https://aistudio.google.com/apikey |

**Locally**: copy `.env.example` to `.env` and fill in your keys.
In **Colab**: add them in the *Secrets* panel (🔑).

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if IN_COLAB:
    from google.colab import userdata
    for key in ("ANTHROPIC_API_KEY", "OPENAI_API_KEY", "GOOGLE_API_KEY"):
        try:
            value = userdata.get(key)
            if value:
                os.environ[key] = value
        except Exception:
            pass

PROVIDERS = {
    "anthropic": bool(os.getenv("ANTHROPIC_API_KEY")),
    "openai": bool(os.getenv("OPENAI_API_KEY")),
    "google": bool(os.getenv("GOOGLE_API_KEY")),
}

for name, present in PROVIDERS.items():
    print(("✅" if present else "⚪") + f" {name}: " + ("key detected" if present else "no key"))

if not any(PROVIDERS.values()):
    print("\n❌ No key detected. Copy .env.example to .env and fill in at least one.")

## 4 · Test call to each provider

A minimal call (cost under 0.001 EUR) to verify that the key really works.

In [ ]:
PROMPT = "Reply with only the word: ready"

def check_anthropic():
    import anthropic
    client = anthropic.Anthropic()
    msg = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=16,
        messages=[{"role": "user", "content": PROMPT}],
    )
    return next(b.text for b in msg.content if b.type == "text")

def check_openai():
    from openai import OpenAI
    client = OpenAI()
    resp = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[{"role": "user", "content": PROMPT}],
    )
    return resp.choices[0].message.content

def check_google():
    from google import genai
    client = genai.Client()
    resp = client.models.generate_content(model="gemini-flash-latest", contents=PROMPT)
    return resp.text

CHECKS = {"anthropic": check_anthropic, "openai": check_openai, "google": check_google}
API_OK = {}

for name, fn in CHECKS.items():
    if not PROVIDERS[name]:
        print(f"⚪ {name}: no key, skipped")
        continue
    try:
        answer = (fn() or "").strip()
        API_OK[name] = True
        print(f"✅ {name}: responds -> {answer!r}")
    except Exception as exc:
        API_OK[name] = False
        print(f"❌ {name}: {type(exc).__name__}: {exc}")

api_ok = any(API_OK.values())

## 5 · Optional tools

- **git**: needed to clone the course repository.
- **Docker**: optional, only for one Session 3 demo. Not essential.

In [ ]:
import shutil, subprocess

for tool, icon_missing, note in [("git", "⚠️", "needed to clone the repo"),
                                 ("docker", "⚪", "optional, Session 3 only")]:
    if shutil.which(tool):
        version = subprocess.run([tool, "--version"], capture_output=True, text=True).stdout.strip()
        print(f"✅ {version}")
    else:
        print(f"{icon_missing} {tool}: not found ({note})")

## Final result

In [ ]:
print("=" * 50)
if python_ok and packages_ok and api_ok:
    print("🟢 ALL SET. See you in Session 1 (Monday 13, 10:00).")
else:
    print("🔴 Something is still pending:")
    if not python_ok:
        print("   - Python: install 3.11 or higher (section 1)")
    if not packages_ok:
        print("   - Packages: pip install -r requirements.txt (section 2)")
    if not api_ok:
        print("   - API: configure at least one working key (sections 3 and 4)")
    print("   If you get stuck: chema@montevive.ai")
print("=" * 50)

---



**Montevive AI** · [montevive.ai](https://montevive.ai) · chema@montevive.ai
Course for Fundacion AI Granada Research & Innovation · July 2026